# Modèle TF-IDF + Logistic Regression

## Importations

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, precision_recall_curve, auc

## Charger les données nettoyées 

In [ ]:
train_path = "data/train_cleaned.csv"
test_path = "data/test_cleaned.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

X_train = df_train['cleaned_text'].values
y_train = df_train['label'].values

X_test = df_test['cleaned_text'].values
y_test = df_test['label'].values

## Vectorisation TF-IDF + GridSearchCV

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

model = LogisticRegression(random_state=42)
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_macro')
grid_search.fit(X_train_vec, y_train)

## Évaluation

In [ ]:
y_pred = grid_search.predict(X_test_vec)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"Best params: {grid_search.best_params_}")
print(f"F1-score: {f1:.3f}")

## Sauvegarde des prédictions

In [ ]:
os.makedirs("predictions", exist_ok=True)

df_result = pd.DataFrame({
    'id': df_test['id'],
    'y_true': y_test,
    'y_pred': y_pred
})
df_result.to_csv("predictions/tfidf_seed42.csv", index=False)

print("✅ Prédictions sauvegardées dans `predictions/tfidf_seed42.csv`.")